### **Imports, and Configurations**

In [2]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import ast, gc, hashlib, json, os, re, tempfile, warnings

import numpy as np
import pandas as pd
import sacrebleu
import torch

from IPython.display import display
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

warnings.filterwarnings("ignore")

PROJECT_DIR = Path("/home/mabdallah/alexandriax_mt_14d")

BASE_MODEL_DIR = PROJECT_DIR / "models/hf/NileChat-3B-Base"
OLD_RUN_DIR = (
    PROJECT_DIR / "runs/nilechat3b_all14/"
    "nilechat3b_alexandria_all14_context3_complete2shot_"
    "all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1"
)
SELECTION_RUN_DIR = (
    PROJECT_DIR / "checkpoint_selection/"
    "nilechat3b_dev12250_hftest60_from16600_complete2shot_"
    "r16_alpha32_2epochs_nonquant_server5090_v1_beam4_training_style"
)
SELECTION_RESULTS_DIR = SELECTION_RUN_DIR / "results"

STAGE_ROOT = PROJECT_DIR / "variant_selection_competition_v1"
CACHE_DIR = STAGE_ROOT / "_shared_cache"
STAGE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set these only if automatic discovery cannot locate the two saved splits.
LOCKED40_PATH = None
PUBLIC60_PATH = None

ORIGINAL_TRAIN_PATH = (
    PROJECT_DIR / "inference_variants/_shared_cache/"
    "paired_dev_train_v3/train_fewshot_pool_df.pkl"
)
OFFICIAL_DEV_PATH = (
    PROJECT_DIR / "inference_variants/_shared_cache/"
    "paired_dev_train_v3/official_dev_df.pkl"
)

COUNTRIES = ["EG", "JO", "LB", "LY", "MA", "MR", "OM",
             "PS", "SA", "SD", "SY", "TN", "YE"]

SPECIALISTS = {
    "cont_4900": ["MA", "OM", "YE"],
    "cont_6400": ["LY"],
    "cont_2000": ["EG", "MR"],
    "cont_1600": ["SA"],
    "cont_7600": ["TN"],
}

LEGACY_PROXY = {"LY": "TN", "SD": "EG"}

MAX_CONTEXT_TURNS = 3
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 120
GEN_BATCH_SIZE = 2
SAVE_EVERY = 100
N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

GENERATION_KWARGS = dict(
    do_sample=False,
    num_beams=4,
    num_return_sequences=1,
    length_penalty=1.0,
    early_stopping=True,
    repetition_penalty=1.05,
    use_cache=True,
)

RETRIEVER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RETRIEVER_BATCH_SIZE = 256

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

print("Experiment root:", STAGE_ROOT)

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1


### **Data and Prompts Preparation**

In [3]:
def read_table(path):
    path = Path(path)
    if path.suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix in {".jsonl", ".json"}:
        return pd.read_json(path, lines=path.suffix == ".jsonl")
    raise ValueError(path)


ALIASES = {
    "country": ["country", "config"],
    "source_id": ["source_id", "id", "turn_id"],
    "conversation_id": ["conversation_id", "conversation", "dialogue_id"],
    "turn_order": ["turn_order", "turn_id_in_conversation", "turn_index"],
    "source_text": ["source_text", "english", "source", "input_text"],
    "reference_arabic": [
        "reference_arabic", "target_arabic", "reference", "target"
    ],
    "dialect": ["dialect", "target_dialect"],
    "domain": ["domain"],
    "participants": ["participants", "roles", "persona"],
    "speaker": ["speaker", "current_speaker"],
    "gender_direction": ["gender_direction", "gender"],
    "previous_english_turns": [
        "previous_english_turns", "previous_turns", "context"
    ],
}


def normalize_frame(df):
    df = df.copy()

    for canonical, candidates in ALIASES.items():
        if canonical not in df:
            for candidate in candidates:
                if candidate in df:
                    df[canonical] = df[candidate]
                    break

    defaults = {
        "dialect": "", "domain": "", "participants": "",
        "speaker": "", "gender_direction": "",
        "previous_english_turns": [],
    }

    for column, default in defaults.items():
        if column not in df:
            df[column] = [default] * len(df) if isinstance(default, list) else default

    if "country" in df:
        df["country"] = df["country"].astype(str).str.strip().str.upper()

    return df


def discover_exact_frame(expected_rows, override=None):
    if override:
        df = normalize_frame(read_table(override))
        if len(df) != expected_rows:
            raise RuntimeError(f"{override}: expected {expected_rows}, found {len(df)}")
        return df, Path(override)

    roots = [
        SELECTION_RUN_DIR,
        PROJECT_DIR / "data",
        PROJECT_DIR / "checkpoint_selection",
        PROJECT_DIR / "runs",
    ]

    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for suffix in ("*.pkl", "*.parquet", "*.csv", "*.jsonl"):
            for path in root.rglob(suffix):
                if STAGE_ROOT in path.parents:
                    continue
                try:
                    df = normalize_frame(read_table(path))
                except Exception:
                    continue

                required = {"country", "source_id", "source_text", "reference_arabic"}
                if len(df) == expected_rows and required.issubset(df.columns):
                    richness = sum(
                        c in df for c in [
                            "conversation_id", "turn_order", "dialect", "domain",
                            "speaker", "gender_direction",
                            "previous_english_turns", "participants",
                        ]
                    )
                    candidates.append((richness, path, df))

    if not candidates:
        raise FileNotFoundError(
            f"Could not automatically locate the saved {expected_rows:,}-row frame. "
            "Set its override path in Cell 2."
        )

    candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
    _, path, df = candidates[0]
    return df, path


locked40_df, locked40_source = discover_exact_frame(5772, LOCKED40_PATH)

try:
    public60_df, public60_source = discover_exact_frame(8670, PUBLIC60_PATH)
except FileNotFoundError:
    full_public_df, full_public_source = discover_exact_frame(14442)

    heldout_ids = set(locked40_df["source_id"].astype(str))
    public60_df = full_public_df[
        ~full_public_df["source_id"].astype(str).isin(heldout_ids)
    ].copy()
    public60_source = full_public_source

    if len(public60_df) != 8670:
        raise RuntimeError(
            f"Subtracting heldout40 produced {len(public60_df)}, expected 8,670."
        )

original_train_df = normalize_frame(read_table(ORIGINAL_TRAIN_PATH))
official_dev_df = normalize_frame(read_table(OFFICIAL_DEV_PATH))

for frame in (original_train_df, official_dev_df, public60_df):
    if "target_arabic" not in frame:
        frame["target_arabic"] = frame["reference_arabic"]

locked40_df = locked40_df.reset_index(drop=True)
locked40_df["_row_order"] = np.arange(len(locked40_df))

assert len(locked40_df) == 5772
assert len(public60_df) == 8670
assert set(locked40_df["country"]) == set(COUNTRIES)
assert locked40_df["source_id"].astype(str).is_unique

gain_df = pd.read_csv(
    SELECTION_RESULTS_DIR / "checkpoint_country_gains_vs_step5200.csv"
)

ADAPTER_PATHS = {}

for step in [5200, 4900, 6400, 2000, 1600, 7600]:
    key = f"cont_{step}"
    rows = gain_df[
        (gain_df["best_step"] == step) |
        (gain_df["default_step"] == step)
    ]

    paths = []
    for column in ["best_checkpoint_path", "default_checkpoint_path"]:
        if column in rows:
            paths += rows[column].dropna().astype(str).tolist()

    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError(f"Saved adapter path not found for step {step}")
    ADAPTER_PATHS[key] = existing[0]

for step in [16000, 16500, 16600]:
    path = OLD_RUN_DIR / f"checkpoint-{step}"
    if not path.exists():
        raise FileNotFoundError(path)
    ADAPTER_PATHS[f"old_{step}"] = path

print("Locked 40%:", locked40_source, locked40_df.shape)
print("Public 60%:", public60_source, public60_df.shape)
print("Original train:", original_train_df.shape)
print("Official DEV:", official_dev_df.shape)
display(pd.Series({k: str(v) for k, v in ADAPTER_PATHS.items()},
                  name="adapter_path").to_frame())

Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


In [4]:
def clean_string(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def parse_list(value):
    if isinstance(value, (list, tuple)):
        return list(value)
    if not clean_string(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return list(parsed) if isinstance(parsed, (list, tuple)) else [parsed]
    except Exception:
        return [str(value)]


def frame_fingerprint(df, columns):
    values = df[columns].fillna("").astype(str)
    payload = pd.util.hash_pandas_object(values, index=False).values.tobytes()
    return hashlib.sha256(payload).hexdigest()


def prepare_pool(df, origin):
    df = normalize_frame(df)
    if "target_arabic" not in df:
        df["target_arabic"] = df["reference_arabic"]

    df = df.copy()
    df["pool_origin"] = origin
    df["source_id"] = df["source_id"].astype(str)
    df["conversation_id"] = df["conversation_id"].astype(str)
    df["source_text"] = df["source_text"].fillna("").astype(str)
    df["target_arabic"] = df["target_arabic"].fillna("").astype(str)
    return df


new_pool_df = pd.concat([
    prepare_pool(original_train_df, "original_train"),
    prepare_pool(official_dev_df, "official_dev"),
    prepare_pool(public60_df, "public60"),
], ignore_index=True)

heldout_ids = set(locked40_df["source_id"].astype(str))
heldout_conversations = set(
    zip(
        locked40_df["country"].astype(str),
        locked40_df["conversation_id"].astype(str),
    )
)

new_pool_df = new_pool_df[
    ~new_pool_df["source_id"].astype(str).isin(heldout_ids)
].copy()

new_pool_df = new_pool_df[
    ~new_pool_df.apply(
        lambda r: (str(r["country"]), str(r["conversation_id"]))
                  in heldout_conversations,
        axis=1,
    )
].copy()

new_pool_df["fewshot_total_chars"] = (
    new_pool_df["source_text"].str.len()
    + new_pool_df["target_arabic"].str.len()
)
new_pool_df = new_pool_df[
    new_pool_df["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS
].drop_duplicates(
    ["pool_origin", "country", "source_id"]
).reset_index(drop=True)

legacy_pool_df = new_pool_df[
    new_pool_df["pool_origin"] == "original_train"
].reset_index(drop=True)

embedding_path = CACHE_DIR / "retrieval_embeddings.npz"
embedding_meta_path = CACHE_DIR / "retrieval_embeddings_meta.json"

embedding_fingerprint = hashlib.sha256(
    (
        frame_fingerprint(
            new_pool_df,
            ["pool_origin", "country", "source_id", "source_text"],
        )
        + frame_fingerprint(locked40_df, ["country", "source_id", "source_text"])
        + RETRIEVER_MODEL_NAME
    ).encode()
).hexdigest()

if (
    embedding_path.exists()
    and embedding_meta_path.exists()
    and json.loads(embedding_meta_path.read_text())["fingerprint"]
        == embedding_fingerprint
):
    cached = np.load(embedding_path)
    pool_embeddings = cached["pool"]
    query_embeddings = cached["query"]
    print("Reused cached retrieval embeddings.")
else:
    retriever = SentenceTransformer(RETRIEVER_MODEL_NAME)

    pool_embeddings = retriever.encode(
        new_pool_df["source_text"].tolist(),
        batch_size=RETRIEVER_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    query_embeddings = retriever.encode(
        locked40_df["source_text"].tolist(),
        batch_size=RETRIEVER_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    np.savez_compressed(
        embedding_path,
        pool=pool_embeddings,
        query=query_embeddings,
    )
    embedding_meta_path.write_text(
        json.dumps({"fingerprint": embedding_fingerprint}, indent=2)
    )
    del retriever
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pool_position = {
    (row.pool_origin, str(row.country), str(row.source_id)): i
    for i, row in new_pool_df.iterrows()
}

legacy_positions = np.array([
    pool_position[(r.pool_origin, str(r.country), str(r.source_id))]
    for _, r in legacy_pool_df.iterrows()
])


def select_few_shots(mode):
    cache_path = CACHE_DIR / f"selected_few_shots_{mode}.pkl"
    fingerprint_path = CACHE_DIR / f"selected_few_shots_{mode}.json"

    fingerprint = hashlib.sha256(
        f"{embedding_fingerprint}|{mode}|{LEGACY_PROXY}|v1".encode()
    ).hexdigest()

    if cache_path.exists() and fingerprint_path.exists():
        metadata = json.loads(fingerprint_path.read_text())
        if metadata["fingerprint"] == fingerprint:
            cached_df = pd.read_pickle(cache_path)
            print(f"Reused cached {mode} selections.")
            return dict(zip(cached_df["source_id"], cached_df["few_shot_examples"]))

    pool = new_pool_df if mode == "new" else legacy_pool_df
    positions = np.arange(len(new_pool_df)) if mode == "new" else legacy_positions
    results = []

    for query_index, query in tqdm(
        locked40_df.iterrows(),
        total=len(locked40_df),
        desc=f"Selecting {mode} examples",
    ):
        query_country = str(query["country"])
        retrieval_country = (
            LEGACY_PROXY.get(query_country, query_country)
            if mode == "legacy"
            else query_country
        )

        same_country = pool["country"].astype(str).eq(retrieval_country)
        same_domain = pool["domain"].astype(str).eq(str(query["domain"]))

        candidate_mask = same_country & same_domain
        if candidate_mask.sum() < N_FEW_SHOTS:
            candidate_mask = same_country

        candidate_local = np.flatnonzero(candidate_mask.to_numpy())
        if len(candidate_local) < N_FEW_SHOTS:
            raise RuntimeError(
                f"Insufficient {mode} retrieval examples for {query_country}"
            )

        candidate_global = positions[candidate_local]
        candidate_df = pool.iloc[candidate_local]

        scores = pool_embeddings[candidate_global] @ query_embeddings[query_index]

        scores += (
            candidate_df["dialect"].astype(str).to_numpy()
            == str(query["dialect"])
        ) * 0.05
        scores += (
            candidate_df["gender_direction"].astype(str).to_numpy()
            == str(query["gender_direction"])
        ) * 0.03
        scores += (
            candidate_df["speaker"].astype(str).to_numpy()
            == str(query["speaker"])
        ) * 0.02

        order = np.argsort(-scores)
        selected = []
        used_conversations = set()
        used_sources = set()

        for ranked_index in order:
            record = candidate_df.iloc[ranked_index]
            conversation_key = (
                str(record["country"]),
                str(record["conversation_id"]),
            )
            source_key = (
                str(record["pool_origin"]),
                str(record["country"]),
                str(record["source_id"]),
            )

            if source_key in used_sources or conversation_key in used_conversations:
                continue

            selected.append(record.to_dict())
            used_sources.add(source_key)
            used_conversations.add(conversation_key)

            if len(selected) == N_FEW_SHOTS:
                break

        if len(selected) < N_FEW_SHOTS:
            for ranked_index in order:
                record = candidate_df.iloc[ranked_index]
                source_key = (
                    str(record["pool_origin"]),
                    str(record["country"]),
                    str(record["source_id"]),
                )
                if source_key in used_sources:
                    continue
                selected.append(record.to_dict())
                used_sources.add(source_key)
                if len(selected) == N_FEW_SHOTS:
                    break

        results.append({
            "source_id": str(query["source_id"]),
            "few_shot_examples": selected,
        })

    selected_df = pd.DataFrame(results)
    selected_df.to_pickle(cache_path)
    fingerprint_path.write_text(
        json.dumps({"fingerprint": fingerprint}, indent=2)
    )
    return dict(zip(selected_df["source_id"],
                    selected_df["few_shot_examples"]))


SHOT_MAPS = {
    "new": select_few_shots("new"),
    "legacy": select_few_shots("legacy"),
}

assert all(len(v) == 2 for mapping in SHOT_MAPS.values() for v in mapping.values())
print("Cached selections:", {k: len(v) for k, v in SHOT_MAPS.items()})

Batches:   0%|          | 0/341 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Selecting new examples:   0%|          | 0/5772 [00:00<?, ?it/s]

Selecting legacy examples:   0%|          | 0/5772 [00:00<?, ?it/s]

Cached selections: {'new': 5772, 'legacy': 5772}


In [5]:
def format_previous_context(row, include_speakers):
    turns = parse_list(row["previous_english_turns"])[-MAX_CONTEXT_TURNS:]
    if not turns:
        return "No previous context."

    lines = []
    for i, turn in enumerate(turns, 1):
        if isinstance(turn, dict):
            text = clean_string(
                turn.get("source_text", turn.get("text", turn.get("english", "")))
            )
            speaker = clean_string(turn.get("speaker", ""))
        else:
            text, speaker = clean_string(turn), ""

        lines.append(
            f"{i}. {speaker}: {text}"
            if include_speakers and speaker
            else f"{i}. {text}"
        )
    return "\n".join(lines)


def format_example(example, index):
    metadata = [
        f"config={clean_string(example.get('country', example.get('config', '')))}",
        f"dialect={clean_string(example.get('dialect', ''))}",
        f"domain={clean_string(example.get('domain', ''))}",
    ]
    return (
        f"Example {index} ({', '.join(metadata)})\n"
        f"English:\n{clean_string(example.get('source_text'))}\n\n"
        f"Arabic:\n{clean_string(example.get('target_arabic'))}"
    )


def build_prompt(row, variant, retrieval_mode):
    include_participants = variant == "V05"
    include_speakers = variant in {"V03", "V05"}

    examples = SHOT_MAPS[retrieval_mode][str(row["source_id"])]
    example_text = "\n\n".join(
        format_example(example, i)
        for i, example in enumerate(examples, 1)
    )

    metadata = [
        f"Country/config: {clean_string(row['country'])}",
        f"Target dialect: {clean_string(row['dialect'])}",
        f"Domain: {clean_string(row['domain'])}",
    ]

    if include_participants and clean_string(row["participants"]):
        metadata.append(f"Persona/Roles: {clean_string(row['participants'])}")
    if clean_string(row["speaker"]):
        metadata.append(f"Current speaker: {clean_string(row['speaker'])}")
    if clean_string(row["gender_direction"]):
        metadata.append(
            "Speaker-to-addressee gender direction: "
            f"{clean_string(row['gender_direction'])}"
        )

    return f"""### System:
{SYSTEM_PROMPT}

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{example_text}

Metadata:
{chr(10).join(metadata)}

Previous English dialogue context:
{format_previous_context(row, include_speakers)}

Current English turn:
{clean_string(row["source_text"])}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation.

### Arabic translation:
"""


def clean_prediction(text):
    text = clean_string(text)
    for token in [
        "<|endoftext|>", "<|im_end|>", "<|im_start|>", "<turn|>",
        getattr(tokenizer, "eos_token", None),
        getattr(tokenizer, "pad_token", None),
    ]:
        if token:
            text = text.replace(token, "")

    for marker in [
        "### Arabic translation:", "Arabic translation:",
        "### Arabic:", "Arabic:", "الترجمة العربية:",
    ]:
        if marker in text:
            text = text.rsplit(marker, 1)[-1]

    blocked = (
        "return only", "do not", "don't add", "preserve", "use the",
        "task:", "rules:", "metadata:", "previous english",
        "current english", "few-shot", "example", "english:",
        "arabic translation", "translation:", "target dialect",
        "speaker-to-addressee",
    )

    kept = []
    for line in text.replace("```", "").splitlines():
        line = line.strip(" #`\t")
        if not line or line.lower().startswith(blocked):
            continue

        letters = [c for c in line if c.isalpha()]
        if letters:
            arabic_ratio = sum("\u0600" <= c <= "\u06ff" for c in letters) / len(letters)
            latin_ratio = sum(("a" <= c.lower() <= "z") for c in letters) / len(letters)
            if arabic_ratio < 0.10 and latin_ratio > 0.35:
                continue
        kept.append(line)

    return " ".join(kept).strip()


def atomic_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".csv", delete=False,
        dir=path.parent, encoding="utf-8"
    ) as handle:
        temp_path = Path(handle.name)
        df.to_csv(handle, index=False)
    os.replace(temp_path, path)


def activate_adapter(adapter_key):
    path = str(ADAPTER_PATHS[adapter_key])
    loaded = set(getattr(model, "peft_config", {}).keys())

    if adapter_key not in loaded:
        model.load_adapter(path, adapter_name=adapter_key, is_trainable=False)

    model.set_adapter(adapter_key)
    model.eval()
    return path


@torch.inference_mode()
def generate_batch(prompts):
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    device = next(model.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]

    output = model.generate(
        **encoded,
        max_new_tokens=MAX_NEW_TOKENS,
        **GENERATION_KWARGS,
    )

    generated = output[:, prompt_length:]
    decoded = tokenizer.batch_decode(generated, skip_special_tokens=False)
    return [clean_prediction(text) for text in decoded]


def score_predictions(predictions, experiment_name):
    predictions = predictions.copy()
    predictions["source_id"] = predictions["source_id"].astype(str)

    references = locked40_df[
        ["source_id", "country", "reference_arabic", "_row_order"]
    ].copy()
    references["source_id"] = references["source_id"].astype(str)

    scored = predictions.drop(
        columns=["country", "reference_arabic", "_row_order"],
        errors="ignore",
    ).merge(
        references,
        on="source_id",
        how="inner",
        validate="one_to_one",
    ).sort_values("_row_order")

    if scored["prediction"].isna().any() or (scored["prediction"].str.len() == 0).any():
        raise RuntimeError(f"{experiment_name} contains empty predictions.")

    records = []
    for country, part in scored.groupby("country", sort=True):
        refs = part["reference_arabic"].astype(str).tolist()
        preds = part["prediction"].astype(str).tolist()

        records.append({
            "experiment": experiment_name,
            "country": country,
            "rows": len(part),
            "spBLEU": sacrebleu.corpus_bleu(
                preds, [refs], tokenize="flores200"
            ).score,
            "chrF++": sacrebleu.corpus_chrf(
                preds, [refs], word_order=2
            ).score,
        })

    metrics = pd.DataFrame(records)
    output_dir = STAGE_ROOT / experiment_name
    atomic_csv(scored, output_dir / "scored_turn_predictions.csv")
    atomic_csv(metrics, output_dir / "per_country_metrics.csv")

    return scored, metrics


def find_saved_v01(step, required_ids):
    pattern = re.compile(
        rf"(?:devft_step_0*{step}|sweep_predictions_step0*{step}|"
        rf"checkpoint-0*{step})(?!\d)",
        flags=re.IGNORECASE,
    )

    roots = [SELECTION_RUN_DIR, ADAPTER_PATHS[f"cont_{step}"].parent]
    candidates = []

    for root in roots:
        for suffix in ("*.csv", "*.parquet", "*.pkl"):
            for path in root.rglob(suffix):
                if STAGE_ROOT in path.parents or not pattern.search(str(path)):
                    continue

                try:
                    frame = normalize_frame(read_table(path))
                except Exception:
                    continue

                prediction_column = next(
                    (c for c in [
                        "prediction", "generated_translation",
                        "predicted_arabic", "clean_prediction",
                    ] if c in frame),
                    None,
                )
                if prediction_column is None or "source_id" not in frame:
                    continue

                frame["source_id"] = frame["source_id"].astype(str)
                coverage = len(required_ids & set(frame["source_id"]))
                if coverage == len(required_ids):
                    quality = (
                        ("training_style" in str(path))
                        + ("sweep_predictions" in path.name)
                        + ("scored" in path.name)
                    )
                    candidates.append((quality, path, frame, prediction_column))

    if not candidates:
        raise FileNotFoundError(
            f"Could not locate complete saved V01 predictions for step {step}."
        )

    candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
    _, path, frame, prediction_column = candidates[0]

    result = frame[["source_id", prediction_column]].rename(
        columns={prediction_column: "prediction"}
    )
    result = result.drop_duplicates("source_id")
    return result, path


def run_variant(
    experiment_name,
    adapter_key,
    variant,
    countries,
    retrieval_mode="new",
    reuse_v01=False,
):
    output_dir = STAGE_ROOT / experiment_name
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / "turn_predictions.csv"

    target = locked40_df[
        locked40_df["country"].isin(countries)
    ].sort_values("_row_order").copy()
    required_ids = set(target["source_id"].astype(str))

    if prediction_path.exists():
        existing = pd.read_csv(prediction_path)
        existing["source_id"] = existing["source_id"].astype(str)
    else:
        existing = pd.DataFrame(columns=["source_id", "prediction"])

    completed_ids = set(existing["source_id"].astype(str))
    missing = target[
        ~target["source_id"].astype(str).isin(completed_ids)
    ].copy()

    imported_from = None

    if reuse_v01 and len(missing):
        saved, imported_from = find_saved_v01(
            int(adapter_key.split("_")[-1]),
            set(missing["source_id"].astype(str)),
        )
        existing = pd.concat([existing, saved], ignore_index=True)
        existing = existing.drop_duplicates("source_id", keep="last")
        atomic_csv(existing, prediction_path)
        missing = target[
            ~target["source_id"].astype(str).isin(
                set(existing["source_id"].astype(str))
            )
        ]

    if len(missing):
        activate_adapter(adapter_key)
        generated_records = []

        for start in tqdm(
            range(0, len(missing), GEN_BATCH_SIZE),
            desc=experiment_name,
        ):
            batch = missing.iloc[start:start + GEN_BATCH_SIZE]
            prompts = [
                build_prompt(row, variant, retrieval_mode)
                for _, row in batch.iterrows()
            ]
            predictions = generate_batch(prompts)

            for (_, row), prediction in zip(batch.iterrows(), predictions):
                generated_records.append({
                    "source_id": str(row["source_id"]),
                    "prediction": prediction,
                })

            if len(generated_records) >= SAVE_EVERY:
                existing = pd.concat(
                    [existing, pd.DataFrame(generated_records)],
                    ignore_index=True,
                ).drop_duplicates("source_id", keep="last")
                atomic_csv(existing, prediction_path)
                generated_records = []

        if generated_records:
            existing = pd.concat(
                [existing, pd.DataFrame(generated_records)],
                ignore_index=True,
            ).drop_duplicates("source_id", keep="last")
            atomic_csv(existing, prediction_path)

    existing = existing[
        existing["source_id"].isin(required_ids)
    ].drop_duplicates("source_id", keep="last")

    if len(existing) != len(target):
        raise RuntimeError(
            f"{experiment_name}: {len(existing)}/{len(target)} predictions complete."
        )

    atomic_csv(existing, prediction_path)
    scored, metrics = score_predictions(existing, experiment_name)

    manifest = {
        "experiment": experiment_name,
        "adapter_key": adapter_key,
        "adapter_path": str(ADAPTER_PATHS[adapter_key]),
        "variant": variant,
        "countries": countries,
        "retrieval_mode": retrieval_mode,
        "reused_v01": reuse_v01,
        "imported_from": str(imported_from) if imported_from else None,
        "rows": len(scored),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    (output_dir / "variant_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    display(metrics)
    print("Macro spBLEU:", metrics["spBLEU"].mean())
    print("Macro chrF++:", metrics["chrF++"].mean())
    return metrics

### **Base Model Load**

In [6]:
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

if "tokenizer" not in globals():
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            str(BASE_MODEL_DIR),
            use_fast=True,
            local_files_only=True,
            trust_remote_code=True,
            padding_side="left",
            truncation_side="left",
        )
    except AttributeError:
        tokenizer = AutoTokenizer.from_pretrained(
            str(BASE_MODEL_DIR),
            use_fast=True,
            local_files_only=True,
            trust_remote_code=True,
            extra_special_tokens={},
            padding_side="left",
            truncation_side="left",
        )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

if "model" not in globals():
    base_model = AutoModelForCausalLM.from_pretrained(
        str(BASE_MODEL_DIR),
        torch_dtype=dtype,
        device_map="auto",
        local_files_only=True,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    model = PeftModel.from_pretrained(
        base_model,
        str(ADAPTER_PATHS["cont_5200"]),
        adapter_name="cont_5200",
        is_trainable=False,
    )
    del base_model
    print("Loaded base model once with adapter cont_5200.")
elif not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(
        model,
        str(ADAPTER_PATHS["cont_5200"]),
        adapter_name="cont_5200",
        is_trainable=False,
    )
    print("Reused existing base model and attached cont_5200.")
else:
    print("Reusing existing tokenizer and PEFT model.")

model.eval()
activate_adapter("cont_5200")

print("Loaded adapters:", list(model.peft_config))
if torch.cuda.is_available():
    print("Allocated GPU GiB:", torch.cuda.memory_allocated() / 2**30)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 5.880150318145752


### **Experiments**

In [7]:
run_variant("c5200_v01", "cont_5200", "V01", COUNTRIES, reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c5200_v01,EG,447,31.731002,45.487254
1,c5200_v01,JO,441,35.499177,49.122497
2,c5200_v01,LB,442,32.288100,46.000736
3,c5200_v01,LY,444,22.856993,38.586206
4,c5200_v01,MA,447,23.126635,39.613185
5,c5200_v01,MR,444,17.760150,34.387167
6,c5200_v01,OM,448,31.841459,46.527519
7,c5200_v01,PS,443,34.256767,48.258044
8,c5200_v01,SA,445,34.730874,49.580115
9,c5200_v01,SD,442,26.152188,40.973329


Macro spBLEU: 29.942952322676767
Macro chrF++: 44.634356315766986


,experiment,country,rows,spBLEU,chrF++
0,c5200_v01,EG,447,31.731002,45.487254
1,c5200_v01,JO,441,35.499177,49.122497
2,c5200_v01,LB,442,32.288100,46.000736
3,c5200_v01,LY,444,22.856993,38.586206
4,c5200_v01,MA,447,23.126635,39.613185
5,c5200_v01,MR,444,17.760150,34.387167
6,c5200_v01,OM,448,31.841459,46.527519
7,c5200_v01,PS,443,34.256767,48.258044
8,c5200_v01,SA,445,34.730874,49.580115
9,c5200_v01,SD,442,26.152188,40.973329


In [9]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
run_variant("c5200_v03", "cont_5200", "V03", COUNTRIES)

c5200_v03:   0%|          | 0/336 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c5200_v03,EG,447,31.351667,45.436582
1,c5200_v03,JO,441,34.824039,48.726856
2,c5200_v03,LB,442,31.856169,45.753562
3,c5200_v03,LY,444,23.054431,38.766709
4,c5200_v03,MA,447,23.306449,39.774620
5,c5200_v03,MR,444,17.634180,34.222957
6,c5200_v03,OM,448,32.066396,46.832468
7,c5200_v03,PS,443,33.419018,47.792473
8,c5200_v03,SA,445,34.324134,49.344198
9,c5200_v03,SD,442,25.629216,40.542699


Macro spBLEU: 29.707937256955628
Macro chrF++: 44.53697331800974


,experiment,country,rows,spBLEU,chrF++
0,c5200_v03,EG,447,31.351667,45.436582
1,c5200_v03,JO,441,34.824039,48.726856
2,c5200_v03,LB,442,31.856169,45.753562
3,c5200_v03,LY,444,23.054431,38.766709
4,c5200_v03,MA,447,23.306449,39.774620
5,c5200_v03,MR,444,17.634180,34.222957
6,c5200_v03,OM,448,32.066396,46.832468
7,c5200_v03,PS,443,33.419018,47.792473
8,c5200_v03,SA,445,34.324134,49.344198
9,c5200_v03,SD,442,25.629216,40.542699


In [28]:
run_variant("c5200_v05", "cont_5200", "V05", COUNTRIES)

c5200_v05:   0%|          | 0/1636 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c5200_v05,EG,447,31.875345,45.595625
1,c5200_v05,JO,441,34.893868,48.713126
2,c5200_v05,LB,442,32.145954,45.855312
3,c5200_v05,LY,444,23.148371,38.895239
4,c5200_v05,MA,447,23.186176,39.673280
5,c5200_v05,MR,444,17.534489,34.096932
6,c5200_v05,OM,448,32.177414,46.899515
7,c5200_v05,PS,443,33.483064,47.823942
8,c5200_v05,SA,445,34.886770,49.725741
9,c5200_v05,SD,442,25.839282,40.668341


Macro spBLEU: 29.872652764805505
Macro chrF++: 44.603348676207716


,experiment,country,rows,spBLEU,chrF++
0,c5200_v05,EG,447,31.875345,45.595625
1,c5200_v05,JO,441,34.893868,48.713126
2,c5200_v05,LB,442,32.145954,45.855312
3,c5200_v05,LY,444,23.148371,38.895239
4,c5200_v05,MA,447,23.186176,39.673280
5,c5200_v05,MR,444,17.534489,34.096932
6,c5200_v05,OM,448,32.177414,46.899515
7,c5200_v05,PS,443,33.483064,47.823942
8,c5200_v05,SA,445,34.886770,49.725741
9,c5200_v05,SD,442,25.839282,40.668341


In [29]:
run_variant("c4900_v01", "cont_4900", "V01", ["MA", "OM", "YE"], reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c4900_v01,MA,447,23.390521,39.750487
1,c4900_v01,OM,448,32.574507,47.108425
2,c4900_v01,YE,442,25.233358,41.705242


Macro spBLEU: 27.066129064261276
Macro chrF++: 42.85471830346896


,experiment,country,rows,spBLEU,chrF++
0,c4900_v01,MA,447,23.390521,39.750487
1,c4900_v01,OM,448,32.574507,47.108425
2,c4900_v01,YE,442,25.233358,41.705242


In [30]:
run_variant("c4900_v03", "cont_4900", "V03", ["MA", "OM", "YE"])

c4900_v03:   0%|          | 0/669 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c4900_v03,MA,447,23.306449,39.774620
1,c4900_v03,OM,448,32.066396,46.832468
2,c4900_v03,YE,442,24.806697,41.388124


Macro spBLEU: 26.72651414134826
Macro chrF++: 42.665070379698456


,experiment,country,rows,spBLEU,chrF++
0,c4900_v03,MA,447,23.306449,39.774620
1,c4900_v03,OM,448,32.066396,46.832468
2,c4900_v03,YE,442,24.806697,41.388124


In [31]:
run_variant("c4900_v05", "cont_4900", "V05", ["MA", "OM", "YE"])

c4900_v05:   0%|          | 0/669 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c4900_v05,MA,447,23.186176,39.673280
1,c4900_v05,OM,448,32.177414,46.898142
2,c4900_v05,YE,442,25.073482,41.485697


Macro spBLEU: 26.812357319106255
Macro chrF++: 42.685706587393504


,experiment,country,rows,spBLEU,chrF++
0,c4900_v05,MA,447,23.186176,39.673280
1,c4900_v05,OM,448,32.177414,46.898142
2,c4900_v05,YE,442,25.073482,41.485697


In [16]:
run_variant("c6400_v01", "cont_6400", "V01", ["LY"], reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c6400_v01,LY,444,23.357296,39.104943


Macro spBLEU: 23.357296281693902
Macro chrF++: 39.104942507654194


,experiment,country,rows,spBLEU,chrF++
0,c6400_v01,LY,444,23.357296,39.104943


In [17]:
run_variant("c6400_v03", "cont_6400", "V03", ["LY"])

c6400_v03:   0%|          | 0/222 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c6400_v03,LY,444,23.227076,38.828795


Macro spBLEU: 23.227076330943774
Macro chrF++: 38.82879470659062


,experiment,country,rows,spBLEU,chrF++
0,c6400_v03,LY,444,23.227076,38.828795


In [18]:
run_variant("c6400_v05", "cont_6400", "V05", ["LY"])

c6400_v05:   0%|          | 0/222 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c6400_v05,LY,444,23.386062,39.013696


Macro spBLEU: 23.386062255892593
Macro chrF++: 39.013695972458756


,experiment,country,rows,spBLEU,chrF++
0,c6400_v05,LY,444,23.386062,39.013696


In [20]:
run_variant("c2000_v01", "cont_2000", "V01", ["EG", "MR"], reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c2000_v01,EG,447,31.998721,45.780024
1,c2000_v01,MR,444,17.940801,34.338484


Macro spBLEU: 24.969761032943182
Macro chrF++: 40.059253820600446


,experiment,country,rows,spBLEU,chrF++
0,c2000_v01,EG,447,31.998721,45.780024
1,c2000_v01,MR,444,17.940801,34.338484


In [32]:
run_variant("c2000_v03", "cont_2000", "V03", ["EG", "MR"])

c2000_v03:   0%|          | 0/446 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c2000_v03,EG,447,31.744517,45.840440
1,c2000_v03,MR,444,17.763525,34.336576


Macro spBLEU: 24.754020773816645
Macro chrF++: 40.08850800487369


,experiment,country,rows,spBLEU,chrF++
0,c2000_v03,EG,447,31.744517,45.840440
1,c2000_v03,MR,444,17.763525,34.336576


In [33]:
run_variant("c2000_v05", "cont_2000", "V05", ["EG", "MR"])

c2000_v05:   0%|          | 0/446 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c2000_v05,EG,447,31.960828,45.751110
1,c2000_v05,MR,444,17.553215,34.008507


Macro spBLEU: 24.757021480040883
Macro chrF++: 39.87980809726237


,experiment,country,rows,spBLEU,chrF++
0,c2000_v05,EG,447,31.960828,45.751110
1,c2000_v05,MR,444,17.553215,34.008507


In [21]:
run_variant("c1600_v01", "cont_1600", "V01", ["SA"], reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c1600_v01,SA,445,35.028283,49.698319


Macro spBLEU: 35.028282704277
Macro chrF++: 49.69831946117979


,experiment,country,rows,spBLEU,chrF++
0,c1600_v01,SA,445,35.028283,49.698319


In [22]:
run_variant("c1600_v03", "cont_1600", "V03", ["SA"])

c1600_v03:   0%|          | 0/223 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c1600_v03,SA,445,35.24504,49.781741


Macro spBLEU: 35.24503983954736
Macro chrF++: 49.781740736891905


,experiment,country,rows,spBLEU,chrF++
0,c1600_v03,SA,445,35.24504,49.781741


In [23]:
run_variant("c1600_v05", "cont_1600", "V05", ["SA"])

c1600_v05:   0%|          | 0/223 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c1600_v05,SA,445,34.724611,49.489067


Macro spBLEU: 34.72461094143405
Macro chrF++: 49.48906735426605


,experiment,country,rows,spBLEU,chrF++
0,c1600_v05,SA,445,34.724611,49.489067


In [24]:
run_variant("c7600_v01", "cont_7600", "V01", ["TN"], reuse_v01=True)

,experiment,country,rows,spBLEU,chrF++
0,c7600_v01,TN,443,35.229468,47.608618


Macro spBLEU: 35.22946834432437
Macro chrF++: 47.60861832656724


,experiment,country,rows,spBLEU,chrF++
0,c7600_v01,TN,443,35.229468,47.608618


In [25]:
run_variant("c7600_v03", "cont_7600", "V03", ["TN"])

c7600_v03:   0%|          | 0/222 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c7600_v03,TN,443,34.536021,47.040737


Macro spBLEU: 34.53602073270702
Macro chrF++: 47.040736622663594


,experiment,country,rows,spBLEU,chrF++
0,c7600_v03,TN,443,34.536021,47.040737


In [26]:
run_variant("c7600_v05", "cont_7600", "V05", ["TN"])

c7600_v05:   0%|          | 0/222 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

,experiment,country,rows,spBLEU,chrF++
0,c7600_v05,TN,443,34.739317,47.261815


Macro spBLEU: 34.73931720399419
Macro chrF++: 47.261815487915214


,experiment,country,rows,spBLEU,chrF++
0,c7600_v05,TN,443,34.739317,47.261815


In [36]:
# ============================================================
# Frozen System92 — complete routed inference + official scoring
# Requires the setup, retrieval, helper, and model-loading cells
# ============================================================

EXPERIMENT_NAME = "system92_frozen_routed"
OUTPUT_DIR = STAGE_ROOT / EXPERIMENT_NAME
PREDICTION_PATH = OUTPUT_DIR / "turn_predictions.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exact previously selected System92 routing policy
SYSTEM92_ROUTE_GROUPS = [
    {
        "adapter": "old_16000",
        "variant": "V03",
        "countries": ["LB", "LY", "MR", "TN"],
    },
    {
        "adapter": "old_16500",
        "variant": "V03",
        "countries": ["PS", "SA", "YE"],
    },
    {
        "adapter": "old_16600",
        "variant": "V03",
        "countries": ["JO"],
    },
    {
        "adapter": "old_16600",
        "variant": "V05",
        "countries": ["EG", "MA", "OM", "SD", "SY"],
    },
]

routed_countries = sorted(
    country
    for route in SYSTEM92_ROUTE_GROUPS
    for country in route["countries"]
)

if routed_countries != sorted(COUNTRIES):
    raise RuntimeError(
        f"Invalid System92 country coverage: {routed_countries}"
    )

# Resume any predictions already saved by this routed experiment
if PREDICTION_PATH.exists():
    saved_predictions = pd.read_csv(PREDICTION_PATH)
    saved_predictions["source_id"] = (
        saved_predictions["source_id"].astype(str)
    )

    valid_saved = (
        saved_predictions["prediction"].notna()
        & saved_predictions["prediction"].astype(str).str.strip().ne("")
    )
    completed_ids = set(
        saved_predictions.loc[valid_saved, "source_id"]
    )
    print(f"Resuming with {len(completed_ids):,} saved predictions.")
else:
    saved_predictions = pd.DataFrame(
        columns=[
            "source_id",
            "prediction",
            "country",
            "adapter_key",
            "variant",
        ]
    )
    completed_ids = set()

for route in SYSTEM92_ROUTE_GROUPS:
    adapter_key = route["adapter"]
    variant = route["variant"]
    route_countries = route["countries"]

    route_rows = (
        locked40_df[
            locked40_df["country"].isin(route_countries)
            & ~locked40_df["source_id"].astype(str).isin(completed_ids)
        ]
        .sort_values("_row_order")
        .copy()
    )

    if route_rows.empty:
        print(
            f"Already complete: {adapter_key} + {variant} "
            f"→ {route_countries}"
        )
        continue

    activate_adapter(adapter_key)

    print(
        f"\nRunning {adapter_key} + {variant} "
        f"for {route_countries}: {len(route_rows):,} rows"
    )

    pending_records = []

    for start in tqdm(
        range(0, len(route_rows), GEN_BATCH_SIZE),
        desc=f"{adapter_key}-{variant}",
    ):
        batch = route_rows.iloc[start:start + GEN_BATCH_SIZE]

        prompts = [
            build_prompt(
                row=row,
                variant=variant,
                retrieval_mode="legacy",
            )
            for _, row in batch.iterrows()
        ]

        batch_predictions = generate_batch(prompts)

        for (_, row), prediction in zip(
            batch.iterrows(),
            batch_predictions,
        ):
            pending_records.append({
                "source_id": str(row["source_id"]),
                "prediction": prediction,
                "country": str(row["country"]),
                "adapter_key": adapter_key,
                "variant": variant,
            })

        if len(pending_records) >= SAVE_EVERY:
            saved_predictions = pd.concat(
                [
                    saved_predictions,
                    pd.DataFrame(pending_records),
                ],
                ignore_index=True,
            ).drop_duplicates(
                "source_id",
                keep="last",
            )

            atomic_csv(saved_predictions, PREDICTION_PATH)
            completed_ids.update(
                record["source_id"]
                for record in pending_records
            )
            pending_records = []

    if pending_records:
        saved_predictions = pd.concat(
            [
                saved_predictions,
                pd.DataFrame(pending_records),
            ],
            ignore_index=True,
        ).drop_duplicates(
            "source_id",
            keep="last",
        )

        atomic_csv(saved_predictions, PREDICTION_PATH)
        completed_ids.update(
            record["source_id"]
            for record in pending_records
        )

# Validate exact held-out coverage
expected_ids = set(locked40_df["source_id"].astype(str))

saved_predictions = (
    saved_predictions[
        saved_predictions["source_id"].astype(str).isin(expected_ids)
    ]
    .drop_duplicates("source_id", keep="last")
    .copy()
)

missing_ids = expected_ids - set(
    saved_predictions["source_id"].astype(str)
)
extra_ids = set(
    saved_predictions["source_id"].astype(str)
) - expected_ids

if missing_ids or extra_ids or len(saved_predictions) != len(locked40_df):
    raise RuntimeError(
        f"Coverage failure: generated={len(saved_predictions):,}, "
        f"expected={len(locked40_df):,}, "
        f"missing={len(missing_ids):,}, extra={len(extra_ids):,}"
    )

if (
    saved_predictions["prediction"].isna().any()
    or saved_predictions["prediction"].astype(str).str.strip().eq("").any()
):
    raise RuntimeError("Empty System92 predictions were detected.")

atomic_csv(saved_predictions, PREDICTION_PATH)

# Official country-level scoring
scored_system92_df, system92_metrics_df = score_predictions(
    saved_predictions,
    EXPERIMENT_NAME,
)

macro_spbleu = float(system92_metrics_df["spBLEU"].mean())
macro_chrfpp = float(system92_metrics_df["chrF++"].mean())

summary = {
    "experiment": EXPERIMENT_NAME,
    "rows": int(len(scored_system92_df)),
    "countries": int(system92_metrics_df["country"].nunique()),
    "macro_spBLEU": macro_spbleu,
    "macro_chrF++": macro_chrfpp,
    "beam": 4,
    "routes": SYSTEM92_ROUTE_GROUPS,
    "legacy_retrieval_proxies": {
        "LY": "TN",
        "SD": "EG",
    },
}

(OUTPUT_DIR / "system92_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("=" * 90)
print("FROZEN SYSTEM92 — LOCKED PUBLIC 40%")
print("=" * 90)
display(
    system92_metrics_df.sort_values("country").reset_index(drop=True)
)

print(f"Rows:             {len(scored_system92_df):,}")
print(f"Countries:        {system92_metrics_df['country'].nunique()}")
print(f"Macro spBLEU:     {macro_spbleu:.6f}")
print(f"Macro chrF++:     {macro_chrfpp:.6f}")
print(f"Predictions:      {PREDICTION_PATH}")

Resuming with 4,344 saved predictions.
Already complete: old_16000 + V03 → ['LB', 'LY', 'MR', 'TN']
Already complete: old_16500 + V03 → ['PS', 'SA', 'YE']
Already complete: old_16600 + V03 → ['JO']

Running old_16600 + V05 for ['EG', 'MA', 'OM', 'SD', 'SY']: 1,428 rows


old_16600-V05:   0%|          | 0/714 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

FROZEN SYSTEM92 — LOCKED PUBLIC 40%


,experiment,country,rows,spBLEU,chrF++
0,system92_frozen_routed,EG,447,29.267321,44.419753
1,system92_frozen_routed,JO,441,31.594844,47.995382
2,system92_frozen_routed,LB,442,28.989743,43.438011
3,system92_frozen_routed,LY,444,17.785729,35.141363
4,system92_frozen_routed,MA,447,21.067677,38.572199
5,system92_frozen_routed,MR,444,15.560348,32.345170
6,system92_frozen_routed,OM,448,28.896068,44.993322
7,system92_frozen_routed,PS,443,31.339030,46.817522
8,system92_frozen_routed,SA,445,32.260249,48.222466
9,system92_frozen_routed,SD,442,21.792684,38.435474


Rows:             5,772
Countries:        13
Macro spBLEU:     26.996195
Macro chrF++:     42.854913
Predictions:      /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1/system92_frozen_routed/turn_predictions.csv


In [37]:
# ============================================================
# Evaluation identity audit — run before Cell 30
# ============================================================

EXPERIMENT_COUNTRIES = {
    "c5200_v01": COUNTRIES,
    "c5200_v03": COUNTRIES,
    "c5200_v05": COUNTRIES,
    "system92_frozen_routed": COUNTRIES,

    "c4900_v01": ["MA", "OM", "YE"],
    "c4900_v03": ["MA", "OM", "YE"],
    "c4900_v05": ["MA", "OM", "YE"],

    "c6400_v01": ["LY"],
    "c6400_v03": ["LY"],
    "c6400_v05": ["LY"],

    "c2000_v01": ["EG", "MR"],
    "c2000_v03": ["EG", "MR"],
    "c2000_v05": ["EG", "MR"],

    "c1600_v01": ["SA"],
    "c1600_v03": ["SA"],
    "c1600_v05": ["SA"],

    "c7600_v01": ["TN"],
    "c7600_v03": ["TN"],
    "c7600_v05": ["TN"],
}

audit_records = []

for experiment, expected_countries in EXPERIMENT_COUNTRIES.items():
    scored_path = (
        STAGE_ROOT
        / experiment
        / "scored_turn_predictions.csv"
    )

    if not scored_path.exists():
        audit_records.append({
            "experiment": experiment,
            "status": "not_completed",
            "expected_rows": None,
            "actual_rows": None,
            "missing_ids": None,
            "extra_ids": None,
        })
        continue

    expected = (
        locked40_df[
            locked40_df["country"].isin(expected_countries)
        ][
            [
                "source_id",
                "country",
                "reference_arabic",
                "_row_order",
            ]
        ]
        .copy()
        .sort_values("_row_order")
        .reset_index(drop=True)
    )

    actual = pd.read_csv(
        scored_path,
        dtype={"source_id": str, "country": str},
    )

    actual["source_id"] = actual["source_id"].astype(str)
    expected["source_id"] = expected["source_id"].astype(str)

    duplicate_count = int(
        actual["source_id"].duplicated().sum()
    )

    expected_ids = set(expected["source_id"])
    actual_ids = set(actual["source_id"])

    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids

    comparable = (
        actual[
            [
                "source_id",
                "country",
                "reference_arabic",
                "_row_order",
            ]
        ]
        .sort_values("_row_order")
        .reset_index(drop=True)
    )

    same_source_ids = (
        comparable["source_id"].tolist()
        == expected["source_id"].tolist()
    )

    same_countries = (
        comparable["country"].astype(str).tolist()
        == expected["country"].astype(str).tolist()
    )

    same_references = (
        comparable["reference_arabic"]
        .fillna("")
        .astype(str)
        .tolist()
        == expected["reference_arabic"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    passed = all([
        len(actual) == len(expected),
        duplicate_count == 0,
        not missing_ids,
        not extra_ids,
        same_source_ids,
        same_countries,
        same_references,
    ])

    audit_records.append({
        "experiment": experiment,
        "status": "exact_match" if passed else "FAILED",
        "expected_rows": len(expected),
        "actual_rows": len(actual),
        "missing_ids": len(missing_ids),
        "extra_ids": len(extra_ids),
        "duplicate_ids": duplicate_count,
        "same_ordered_source_ids": same_source_ids,
        "same_countries": same_countries,
        "same_references": same_references,
    })

audit_df = pd.DataFrame(audit_records)
display(audit_df)

failed_df = audit_df[
    audit_df["status"] == "FAILED"
]

if len(failed_df):
    raise RuntimeError(
        "Evaluation identity mismatch detected:\n"
        + failed_df.to_string(index=False)
    )

completed_df = audit_df[
    audit_df["status"] == "exact_match"
]

print(
    f"Passed: {len(completed_df)} completed experiments use "
    "their exact expected locked40 rows and references."
)

full_system_df = completed_df[
    completed_df["experiment"].isin([
        "c5200_v01",
        "c5200_v03",
        "c5200_v05",
        "system92_frozen_routed",
    ])
]

if len(full_system_df) == 4:
    assert (
        full_system_df["actual_rows"]
        == len(locked40_df)
    ).all()

    print(
        f"Confirmed: all four complete systems predict the same "
        f"{len(locked40_df):,} evaluation turns."
    )
else:
    print(
        "Some full-system experiments are not completed yet; "
        "rerun this audit after they finish."
    )

,experiment,status,expected_rows,actual_rows,missing_ids,extra_ids,duplicate_ids,same_ordered_source_ids,same_countries,same_references
0,c5200_v01,exact_match,5772,5772,0,0,0,True,True,True
1,c5200_v03,exact_match,5772,5772,0,0,0,True,True,True
2,c5200_v05,exact_match,5772,5772,0,0,0,True,True,True
3,system92_frozen_routed,exact_match,5772,5772,0,0,0,True,True,True
4,c4900_v01,exact_match,1337,1337,0,0,0,True,True,True
5,c4900_v03,exact_match,1337,1337,0,0,0,True,True,True
6,c4900_v05,exact_match,1337,1337,0,0,0,True,True,True
7,c6400_v01,exact_match,444,444,0,0,0,True,True,True
8,c6400_v03,exact_match,444,444,0,0,0,True,True,True
9,c6400_v05,exact_match,444,444,0,0,0,True,True,True


Passed: 19 completed experiments use their exact expected locked40 rows and references.
Confirmed: all four complete systems predict the same 5,772 evaluation turns.


### **LeaderBoard**

In [38]:
# ============================================================
# Cell 30 — Updated experiment and country leaderboards
# Separates full-system results from targeted specialists
# ============================================================

FULL_SYSTEM_EXPERIMENTS = [
    "c5200_v01",
    "c5200_v03",
    "c5200_v05",
    "system92_frozen_routed",
]

SPECIALIST_EXPERIMENTS = [
    f"c{step}_{variant}"
    for step in [4900, 6400, 2000, 1600, 7600]
    for variant in ["v01", "v03", "v05"]
]

PLANNED_EXPERIMENTS = (
    FULL_SYSTEM_EXPERIMENTS
    + SPECIALIST_EXPERIMENTS
)

metric_frames = []
missing_experiments = []

for experiment in PLANNED_EXPERIMENTS:
    metrics_path = (
        STAGE_ROOT
        / experiment
        / "per_country_metrics.csv"
    )

    if not metrics_path.exists():
        missing_experiments.append(experiment)
        continue

    metrics = pd.read_csv(metrics_path)
    metrics["experiment"] = experiment
    metrics["experiment_role"] = (
        "frozen_incumbent"
        if experiment == "system92_frozen_routed"
        else "global_new"
        if experiment.startswith("c5200_")
        else "targeted_specialist"
    )
    metric_frames.append(metrics)

if not metric_frames:
    raise FileNotFoundError(
        "No completed experiment metric files were found."
    )

all_metrics_df = pd.concat(
    metric_frames,
    ignore_index=True,
)

duplicate_mask = all_metrics_df.duplicated(
    ["experiment", "country"],
    keep=False,
)

if duplicate_mask.any():
    raise RuntimeError(
        "Duplicate experiment-country metric rows:\n"
        + all_metrics_df.loc[
            duplicate_mask,
            ["experiment", "country"],
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# Per-country candidate leaderboard
# This is the valid comparison for routing decisions
# ------------------------------------------------------------

country_candidate_leaderboard_df = (
    all_metrics_df
    .sort_values(
        ["country", "spBLEU", "chrF++"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

country_candidate_leaderboard_df[
    "country_rank"
] = (
    country_candidate_leaderboard_df
    .groupby("country")
    .cumcount()
    + 1
)

country_candidate_leaderboard_df[
    "gain_vs_best"
] = (
    country_candidate_leaderboard_df["spBLEU"]
    - country_candidate_leaderboard_df
      .groupby("country")["spBLEU"]
      .transform("max")
)

# ------------------------------------------------------------
# Experiment summaries
# Targeted specialists are kept separate because their macro
# scores cover only selected countries
# ------------------------------------------------------------

experiment_summary_df = (
    all_metrics_df
    .groupby(
        ["experiment", "experiment_role"],
        as_index=False,
    )
    .agg(
        countries=("country", "nunique"),
        rows=("rows", "sum"),
        macro_spBLEU=("spBLEU", "mean"),
        macro_chrFpp=("chrF++", "mean"),
    )
)

experiment_summary_df["country_coverage"] = (
    experiment_summary_df["experiment"].map(
        all_metrics_df.groupby("experiment")["country"]
        .apply(lambda x: ",".join(sorted(x.astype(str))))
    )
)

full_system_leaderboard_df = (
    experiment_summary_df[
        experiment_summary_df["experiment"].isin(
            FULL_SYSTEM_EXPERIMENTS
        )
    ]
    .copy()
)

invalid_full_systems = full_system_leaderboard_df[
    (full_system_leaderboard_df["countries"] != len(COUNTRIES))
    | (full_system_leaderboard_df["rows"] != len(locked40_df))
]

if len(invalid_full_systems):
    raise RuntimeError(
        "Incomplete full-system experiments:\n"
        + invalid_full_systems[
            ["experiment", "countries", "rows"]
        ].to_string(index=False)
    )

full_system_leaderboard_df = (
    full_system_leaderboard_df
    .sort_values(
        ["macro_spBLEU", "macro_chrFpp"],
        ascending=False,
    )
    .reset_index(drop=True)
)

full_system_leaderboard_df[
    "system_rank"
] = np.arange(
    1,
    len(full_system_leaderboard_df) + 1,
)

best_full_spbleu = float(
    full_system_leaderboard_df.iloc[0]["macro_spBLEU"]
)

full_system_leaderboard_df[
    "spBLEU_gap_from_best"
] = (
    full_system_leaderboard_df["macro_spBLEU"]
    - best_full_spbleu
)

targeted_specialist_summary_df = (
    experiment_summary_df[
        experiment_summary_df["experiment_role"]
        == "targeted_specialist"
    ]
    .sort_values(
        ["macro_spBLEU", "macro_chrFpp"],
        ascending=False,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

atomic_csv(
    all_metrics_df,
    STAGE_ROOT / "all_variant_country_metrics.csv",
)

atomic_csv(
    country_candidate_leaderboard_df,
    STAGE_ROOT / "country_candidate_leaderboard.csv",
)

atomic_csv(
    full_system_leaderboard_df,
    STAGE_ROOT / "full_system_experiment_leaderboard.csv",
)

atomic_csv(
    targeted_specialist_summary_df,
    STAGE_ROOT / "targeted_specialist_summary.csv",
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 90)
print("FULL 13-COUNTRY SYSTEM LEADERBOARD")
print("=" * 90)

display(
    full_system_leaderboard_df[
        [
            "system_rank",
            "experiment",
            "experiment_role",
            "rows",
            "macro_spBLEU",
            "macro_chrFpp",
            "spBLEU_gap_from_best",
        ]
    ]
)

print("\nBest candidate independently for each country:")

display(
    country_candidate_leaderboard_df[
        country_candidate_leaderboard_df["country_rank"] == 1
    ][
        [
            "country",
            "experiment",
            "experiment_role",
            "spBLEU",
            "chrF++",
        ]
    ].reset_index(drop=True)
)

print("\nTargeted specialist results:")
display(targeted_specialist_summary_df)

if missing_experiments:
    print("\nExperiments not completed yet:")
    print(", ".join(missing_experiments))

FULL 13-COUNTRY SYSTEM LEADERBOARD


,system_rank,experiment,experiment_role,rows,macro_spBLEU,macro_chrFpp,spBLEU_gap_from_best
0,1,c5200_v01,global_new,5772,29.942952,44.634356,0.000000
1,2,c5200_v05,global_new,5772,29.872653,44.603349,-0.070300
2,3,c5200_v03,global_new,5772,29.707937,44.536973,-0.235015
3,4,system92_frozen_routed,frozen_incumbent,5772,26.996195,42.854913,-2.946757



Best candidate independently for each country:


,country,experiment,experiment_role,spBLEU,chrF++
0,EG,c2000_v01,targeted_specialist,31.998721,45.780024
1,JO,c5200_v01,global_new,35.499177,49.122497
2,LB,c5200_v01,global_new,32.288100,46.000736
3,LY,c6400_v05,targeted_specialist,23.386062,39.013696
4,MA,c4900_v01,targeted_specialist,23.390521,39.750487
5,MR,c2000_v01,targeted_specialist,17.940801,34.338484
6,OM,c4900_v01,targeted_specialist,32.574507,47.108425
7,PS,c5200_v01,global_new,34.256767,48.258044
8,SA,c1600_v03,targeted_specialist,35.245040,49.781741
9,SD,c5200_v01,global_new,26.152188,40.973329



Targeted specialist results:


,experiment,experiment_role,countries,rows,macro_spBLEU,macro_chrFpp,country_coverage
0,c1600_v03,targeted_specialist,1,445,35.245040,49.781741,SA
1,c7600_v01,targeted_specialist,1,443,35.229468,47.608618,TN
2,c1600_v01,targeted_specialist,1,445,35.028283,49.698319,SA
3,c7600_v05,targeted_specialist,1,443,34.739317,47.261815,TN
4,c1600_v05,targeted_specialist,1,445,34.724611,49.489067,SA
5,c7600_v03,targeted_specialist,1,443,34.536021,47.040737,TN
6,c4900_v01,targeted_specialist,3,1337,27.066129,42.854718,"MA,OM,YE"
7,c4900_v05,targeted_specialist,3,1337,26.812357,42.685707,"MA,OM,YE"
8,c4900_v03,targeted_specialist,3,1337,26.726514,42.665070,"MA,OM,YE"
9,c2000_v01,targeted_specialist,2,891,24.969761,40.059254,"EG,MR"


### **Official Submission**

In [39]:
# ============================================================
# Cell 1 — Freeze final grouped TEST system
# ============================================================

import time
import zipfile
from collections import defaultdict
from IPython.display import FileLink

TEST_SYSTEM_NAME = (
    "100_official_private_test_"
    "continuation_router_threshold015_beam4_v1"
)

TEST_OUTPUT_DIR = (
    PROJECT_DIR
    / "inference_variants"
    / TEST_SYSTEM_NAME
)
TEST_CACHE_DIR = (
    PROJECT_DIR
    / "inference_variants"
    / "_shared_cache"
    / "final_continuation_router_test_v1"
)

TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_PREDICTION_PATH = (
    TEST_OUTPUT_DIR / "turn_predictions.csv"
)
TEST_JSONL_PATH = (
    TEST_OUTPUT_DIR / "predictions.jsonl"
)
TEST_ZIP_PATH = (
    TEST_OUTPUT_DIR / "submission_predictions.zip"
)

SELECTION_MACRO_SPBLEU = 30.182420
SELECTION_MACRO_CHRFPP = 44.805134

# V01 = deterministic random training-parity shots
# V03/V05 = expanded semantic-retrieval pool
FINAL_TEST_ROUTES = {
    "EG": {
        "adapter": "cont_5200",
        "variant": "V05",
        "shot_mode": "test_retrieved",
    },
    "JO": {
        "adapter": "cont_5200",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "LB": {
        "adapter": "cont_5200",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "LY": {
        "adapter": "cont_6400",
        "variant": "V05",
        "shot_mode": "test_retrieved",
    },
    "MA": {
        "adapter": "cont_5200",
        "variant": "V03",
        "shot_mode": "test_retrieved",
    },
    "MR": {
        "adapter": "cont_2000",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "OM": {
        "adapter": "cont_4900",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "PS": {
        "adapter": "cont_5200",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "SA": {
        "adapter": "cont_1600",
        "variant": "V03",
        "shot_mode": "test_retrieved",
    },
    "SD": {
        "adapter": "cont_5200",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "SY": {
        "adapter": "cont_5200",
        "variant": "V05",
        "shot_mode": "test_retrieved",
    },
    "TN": {
        "adapter": "cont_7600",
        "variant": "V01",
        "shot_mode": "test_random",
    },
    "YE": {
        "adapter": "cont_4900",
        "variant": "V01",
        "shot_mode": "test_random",
    },
}

if set(FINAL_TEST_ROUTES) != set(COUNTRIES):
    raise RuntimeError("Final route does not cover exactly 13 countries.")

required_adapters = sorted({
    route["adapter"]
    for route in FINAL_TEST_ROUTES.values()
})

for adapter in required_adapters:
    if adapter not in ADAPTER_PATHS:
        raise KeyError(f"Missing adapter registry entry: {adapter}")

    if not (
        ADAPTER_PATHS[adapter] / "adapter_config.json"
    ).exists():
        raise FileNotFoundError(ADAPTER_PATHS[adapter])

required_objects = [
    "model",
    "tokenizer",
    "original_train_df",
    "new_pool_df",
    "pool_embeddings",
    "embedding_fingerprint",
    "SHOT_MAPS",
    "activate_adapter",
    "generate_batch",
    "atomic_csv",
    "frame_fingerprint",
    "format_example",
    "format_previous_context",
    "clean_string",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the existing setup/retrieval/model cells first. Missing: "
        f"{missing_objects}"
    )

# Prevent repetitive pad-token warnings.
GENERATION_KWARGS["pad_token_id"] = tokenizer.pad_token_id
GENERATION_KWARGS["eos_token_id"] = tokenizer.eos_token_id

adapter_order = {
    name: i for i, name in enumerate([
        "cont_5200",
        "cont_4900",
        "cont_6400",
        "cont_2000",
        "cont_1600",
        "cont_7600",
    ])
}

route_df = pd.DataFrame([
    {"country": country, **route}
    for country, route in FINAL_TEST_ROUTES.items()
])

FINAL_TEST_GROUPS = []

for keys, group in route_df.groupby(
    ["adapter", "variant", "shot_mode"],
    sort=False,
):
    adapter, variant, shot_mode = keys

    FINAL_TEST_GROUPS.append({
        "adapter": adapter,
        "variant": variant,
        "shot_mode": shot_mode,
        "countries": sorted(group["country"].tolist()),
    })

FINAL_TEST_GROUPS.sort(
    key=lambda x: (
        adapter_order[x["adapter"]],
        x["variant"],
    )
)

print("=" * 90)
print("FINAL OFFICIAL-TEST SYSTEM FLOW")
print("=" * 90)
print("Base model: loaded once")
print("Inference: grouped by adapter × prompt")
print("Beam: 4")
print("System92 fallback: removed")
print("Expected TEST turns: 14,459")
print("Validated selection spBLEU:", SELECTION_MACRO_SPBLEU)
print("Validated selection chrF++:", SELECTION_MACRO_CHRFPP)
print("Output:", TEST_OUTPUT_DIR)

display(pd.DataFrame(FINAL_TEST_GROUPS))

FINAL OFFICIAL-TEST SYSTEM FLOW
Base model: loaded once
Inference: grouped by adapter × prompt
Beam: 4
System92 fallback: removed
Expected TEST turns: 14,459
Validated selection spBLEU: 30.18242
Validated selection chrF++: 44.805134
Output: /home/mabdallah/alexandriax_mt_14d/inference_variants/100_official_private_test_continuation_router_threshold015_beam4_v1


,adapter,variant,shot_mode,countries
0,cont_5200,V01,test_random,"[JO, LB, PS, SD]"
1,cont_5200,V03,test_retrieved,[MA]
2,cont_5200,V05,test_retrieved,"[EG, SY]"
3,cont_4900,V01,test_random,"[OM, YE]"
4,cont_6400,V05,test_retrieved,[LY]
5,cont_2000,V01,test_random,[MR]
6,cont_1600,V03,test_retrieved,[SA]
7,cont_7600,V01,test_random,[TN]


In [40]:
# ============================================================
# Cell 2 — Load exact Codabench private TEST archive
# ============================================================

PRIVATE_TEST_ARCHIVE = (
    PROJECT_DIR
    / "data"
    / "nilechat3b_all14"
    / "alexandriax_private_test.zip"
)

PRIVATE_TEST_EXTRACT_DIR = (
    TEST_CACHE_DIR
    / "codabench_private_test_807229"
)

EXPECTED_TEST_TURNS = 14459
EXPECTED_TEST_CONVERSATIONS = 4673
EXPECTED_TEST_ID_HASH = (
    "c98770279d64433aa5d7738b8cf155f"
    "41436c0afb612004eac5c4fe93b1a1256"
)

EXPECTED_COUNTRY_TURNS = {
    "EG": 1113,
    "JO": 1109,
    "LB": 1110,
    "LY": 1309,
    "MA": 1111,
    "MR": 1119,
    "OM": 1107,
    "PS": 1111,
    "SA": 1114,
    "SD": 915,
    "SY": 1114,
    "TN": 1114,
    "YE": 1113,
}


def test_plain(value):
    if isinstance(value, np.ndarray):
        return [test_plain(x) for x in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, tuple):
        return [test_plain(x) for x in value]
    if isinstance(value, list):
        return [test_plain(x) for x in value]
    if isinstance(value, dict):
        return {
            str(k): test_plain(v)
            for k, v in value.items()
        }
    return value


def test_text(value):
    value = test_plain(value)

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return str(value).strip()

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalize_test_turns(value):
    value = test_plain(value)

    if value is None:
        return []

    if isinstance(value, list):
        return [
            item if isinstance(item, dict)
            else {"text": test_text(item)}
            for item in value
            if item is not None
        ]

    if isinstance(value, dict):
        list_lengths = [
            len(v) for v in value.values()
            if isinstance(v, list)
        ]

        if not list_lengths:
            return []

        rows = []
        for i in range(max(list_lengths)):
            rows.append({
                key: values[i] if (
                    isinstance(values, list)
                    and i < len(values)
                ) else values
                for key, values in value.items()
            })
        return rows

    return []


def test_turn_field(turn, keys, default=""):
    if not isinstance(turn, dict):
        return default

    for key in keys:
        value = test_text(turn.get(key))
        if value:
            return value

    return default


def test_turn_text(turn):
    return test_turn_field(
        turn,
        [
            "text",
            "source",
            "source_text",
            "english",
            "english_text",
            "sentence",
            "utterance",
            "content",
            "value",
        ],
    )


def test_turn_order(turn, fallback_index):
    value = test_turn_field(
        turn,
        ["turn_order", "turn_id", "order", "idx", "index"],
    )

    try:
        return int(value)
    except Exception:
        return int(fallback_index + 1)


def flatten_test_conversation(record, config):
    conversation_id = test_text(
        record.get(
            "conv_id",
            record.get("conversation_id"),
        )
    )

    if not conversation_id:
        raise RuntimeError(
            f"Missing conversation ID for {config}"
        )

    turns = normalize_test_turns(
        record.get("english_conversation", [])
    )

    if not turns:
        raise RuntimeError(
            f"No English turns in {config}/{conversation_id}"
        )

    records = []

    for turn_index, turn in enumerate(turns):
        source_text = test_turn_text(turn)
        order = test_turn_order(turn, turn_index)

        if not source_text:
            raise RuntimeError(
                f"Empty source: {config}/{conversation_id}/{order}"
            )

        previous = []

        for previous_index in range(
            max(0, turn_index - MAX_CONTEXT_TURNS),
            turn_index,
        ):
            previous_turn = turns[previous_index]

            previous.append({
                "turn_order": test_turn_order(
                    previous_turn,
                    previous_index,
                ),
                "speaker": test_turn_field(
                    previous_turn,
                    [
                        "speaker",
                        "role",
                        "speaker_role",
                        "participant",
                    ],
                ),
                "direction": test_turn_field(
                    previous_turn,
                    [
                        "direction",
                        "gender_direction",
                        "speaker_addressee_gender",
                    ],
                ),
                "text": test_turn_text(previous_turn),
            })

        records.append({
            "source_id": (
                f"{config}_test_"
                f"{conversation_id}_{order}"
            ),
            "config": config,
            "country": config,
            "split": "test",
            "conversation_id": conversation_id,
            "turn_order": int(order),
            "turn_id": int(order),
            "dialect": test_text(
                record.get("dialect", "")
            ),
            "domain": test_text(
                record.get("domain", "")
            ),
            "participants": test_text(
                record.get("participants", "")
            ),
            "speaker": test_turn_field(
                turn,
                [
                    "speaker",
                    "role",
                    "speaker_role",
                    "participant",
                ],
            ),
            "gender_direction": test_turn_field(
                turn,
                [
                    "direction",
                    "gender_direction",
                    "speaker_addressee_gender",
                ],
            ),
            "previous_english_turns": previous,
            "source_text": source_text,
        })

    return records


if not PRIVATE_TEST_ARCHIVE.exists():
    raise FileNotFoundError(
        f"Official TEST archive not found:\n"
        f"{PRIVATE_TEST_ARCHIVE}"
    )

PRIVATE_TEST_EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

source_files = sorted(
    PRIVATE_TEST_EXTRACT_DIR.rglob(
        "alexandria_*_private_test_input.jsonl"
    )
)

if len(source_files) != 13:
    with zipfile.ZipFile(
        PRIVATE_TEST_ARCHIVE,
        "r",
    ) as archive:
        archive.extractall(PRIVATE_TEST_EXTRACT_DIR)

    source_files = sorted(
        PRIVATE_TEST_EXTRACT_DIR.rglob(
            "alexandria_*_private_test_input.jsonl"
        )
    )

if len(source_files) != 13:
    raise RuntimeError(
        f"Expected 13 country JSONL files, found "
        f"{len(source_files)}"
    )

test_records = []

for source_file in source_files:
    match = re.search(
        r"alexandria_(EG|JO|LB|LY|MA|MR|OM|PS|SA|SD|SY|TN|YE)_",
        source_file.name,
        flags=re.IGNORECASE,
    )

    if not match:
        raise RuntimeError(
            f"Cannot resolve country from {source_file.name}"
        )

    config = match.group(1).upper()
    conversation_count = 0

    with open(
        source_file,
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)

            if "english_conversation" not in record:
                for alternative in [
                    "turns",
                    "dialogue",
                    "conversation",
                    "source_conversation",
                ]:
                    if isinstance(
                        record.get(alternative),
                        list,
                    ):
                        record["english_conversation"] = (
                            record[alternative]
                        )
                        break

            test_records.extend(
                flatten_test_conversation(
                    record,
                    config,
                )
            )
            conversation_count += 1

    print(
        f"{config}: "
        f"{conversation_count:,} conversations"
    )

official_test_df = (
    pd.DataFrame(test_records)
    .sort_values(
        ["config", "conversation_id", "turn_order"]
    )
    .reset_index(drop=True)
)

official_test_df["source_id"] = (
    official_test_df["source_id"].astype(str)
)
official_test_df["conversation_id"] = (
    official_test_df["conversation_id"].astype(str)
)
official_test_df["turn_order"] = pd.to_numeric(
    official_test_df["turn_order"],
    errors="raise",
).astype(int)

official_test_df["_test_row_idx"] = np.arange(
    len(official_test_df),
    dtype=np.int64,
)

test_id_hasher = hashlib.sha256()

for source_id in official_test_df["source_id"]:
    test_id_hasher.update(
        str(source_id).encode("utf-8")
    )
    test_id_hasher.update(b"\n")

TEST_ID_HASH = test_id_hasher.hexdigest()

actual_country_turns = (
    official_test_df["config"]
    .value_counts()
    .sort_index()
    .to_dict()
)

actual_conversations = len(
    official_test_df[
        ["config", "conversation_id"]
    ].drop_duplicates()
)

if len(official_test_df) != EXPECTED_TEST_TURNS:
    raise RuntimeError(
        f"Expected {EXPECTED_TEST_TURNS:,} turns, "
        f"found {len(official_test_df):,}"
    )

if actual_conversations != EXPECTED_TEST_CONVERSATIONS:
    raise RuntimeError(
        f"Expected {EXPECTED_TEST_CONVERSATIONS:,} conversations, "
        f"found {actual_conversations:,}"
    )

if actual_country_turns != EXPECTED_COUNTRY_TURNS:
    raise RuntimeError(
        f"Country counts changed:\n{actual_country_turns}"
    )

if TEST_ID_HASH != EXPECTED_TEST_ID_HASH:
    raise RuntimeError(
        f"Wrong TEST revision/hash:\n{TEST_ID_HASH}"
    )

if official_test_df["source_id"].duplicated().any():
    raise RuntimeError("Duplicate TEST source_id.")

if official_test_df[
    ["config", "conversation_id", "turn_order"]
].duplicated().any():
    raise RuntimeError(
        "Duplicate conversation/turn key."
    )

print("\nOFFICIAL PRIVATE TEST CONFIRMED")
print("Turns:", len(official_test_df))
print("Conversations:", actual_conversations)
print("Countries:", sorted(actual_country_turns))
print("ID hash:", TEST_ID_HASH)

display(
    official_test_df.groupby(
        "config",
        as_index=False,
    ).agg(
        conversations=("conversation_id", "nunique"),
        turns=("source_id", "size"),
    )
)

EG: 359 conversations
JO: 348 conversations
LB: 368 conversations
LY: 423 conversations
MA: 362 conversations
MR: 365 conversations
OM: 370 conversations
PS: 351 conversations
SA: 359 conversations
SD: 283 conversations
SY: 356 conversations
TN: 374 conversations
YE: 355 conversations

OFFICIAL PRIVATE TEST CONFIRMED
Turns: 14459
Conversations: 4673
Countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
ID hash: c98770279d64433aa5d7738b8cf155f41436c0afb612004eac5c4fe93b1a1256


,config,conversations,turns
0,EG,359,1113
1,JO,348,1109
2,LB,368,1110
3,LY,423,1309
4,MA,362,1111
5,MR,365,1119
6,OM,370,1107
7,PS,351,1111
8,SA,359,1114
9,SD,283,915


In [41]:
# ============================================================
# Cell 3 — Grouped final inference + diagnostics + submission
# ============================================================

# ------------------------------------------------------------
# Prepare exact V01 deterministic random-shot policy
# ------------------------------------------------------------

FEW_SHOT_COLUMNS = [
    "source_id",
    "config",
    "conversation_id",
    "dialect",
    "domain",
    "participants",
    "speaker",
    "gender_direction",
    "source_text",
    "target_arabic",
]

random_pool = original_train_df.copy()

if "config" not in random_pool:
    random_pool["config"] = random_pool["country"]

for column in FEW_SHOT_COLUMNS:
    if column not in random_pool:
        random_pool[column] = ""

random_pool = (
    random_pool[FEW_SHOT_COLUMNS]
    .copy()
    .reset_index(drop=True)
)

random_pool["fewshot_total_chars"] = (
    random_pool["source_text"].astype(str).str.len()
    + random_pool["target_arabic"].astype(str).str.len()
)

short_random_pool = (
    random_pool[
        random_pool["fewshot_total_chars"]
        <= MAX_FEW_SHOT_EXAMPLE_CHARS
    ]
    .copy()
    .reset_index(drop=True)
)

random_by_config = {
    key: group
    for key, group in random_pool.groupby(
        "config",
        sort=False,
    )
}
random_short_by_config = {
    key: group
    for key, group in short_random_pool.groupby(
        "config",
        sort=False,
    )
}
random_by_config_domain = {
    key: group
    for key, group in random_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}
random_short_by_config_domain = {
    key: group
    for key, group in short_random_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}


def stable_test_seed(source_id, base_seed=3407):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(
        hashlib.md5(raw).hexdigest()[:8],
        16,
    )


def select_test_random_shots(row, n=2):
    config = clean_string(row["config"])
    domain = clean_string(row["domain"])

    candidate_pools = [
        random_short_by_config_domain.get(
            (config, domain)
        ),
        random_short_by_config.get(config),
        short_random_pool,
        random_by_config_domain.get(
            (config, domain)
        ),
        random_by_config.get(config),
        random_pool,
    ]

    for candidates in candidate_pools:
        if candidates is None or len(candidates) == 0:
            continue

        return (
            candidates.sample(
                n=min(n, len(candidates)),
                random_state=stable_test_seed(
                    row["source_id"]
                ),
            )[FEW_SHOT_COLUMNS]
            .to_dict("records")
        )

    raise RuntimeError(
        f"No random shots for {row['source_id']}"
    )


random_cache_path = (
    TEST_CACHE_DIR
    / "selected_test_random_shots.pkl"
)
random_meta_path = (
    TEST_CACHE_DIR
    / "selected_test_random_shots.json"
)

random_fingerprint = hashlib.sha256(
    (
        TEST_ID_HASH
        + frame_fingerprint(
            random_pool,
            [
                "source_id",
                "config",
                "domain",
                "source_text",
                "target_arabic",
            ],
        )
        + "exact_v01_random_policy_v1"
    ).encode()
).hexdigest()

if (
    random_cache_path.exists()
    and random_meta_path.exists()
    and json.loads(
        random_meta_path.read_text()
    )["fingerprint"] == random_fingerprint
):
    random_shots_df = pd.read_pickle(
        random_cache_path
    )
    print("Reused cached V01 TEST shots.")
else:
    random_shots_df = pd.DataFrame([
        {
            "source_id": str(row["source_id"]),
            "few_shot_examples": (
                select_test_random_shots(row)
            ),
        }
        for _, row in tqdm(
            official_test_df.iterrows(),
            total=len(official_test_df),
            desc="V01 deterministic shots",
        )
    ])

    random_shots_df.to_pickle(
        random_cache_path
    )
    random_meta_path.write_text(
        json.dumps(
            {"fingerprint": random_fingerprint},
            indent=2,
        )
    )

test_random_map = dict(zip(
    random_shots_df["source_id"].astype(str),
    random_shots_df["few_shot_examples"],
))

# ------------------------------------------------------------
# Prepare V03/V05 semantic retrieval for TEST
# ------------------------------------------------------------

retrieval_pool = new_pool_df.copy()

if "config" not in retrieval_pool:
    retrieval_pool["config"] = (
        retrieval_pool["country"]
    )

test_embedding_path = (
    TEST_CACHE_DIR
    / "test_query_embeddings.npy"
)
test_embedding_meta_path = (
    TEST_CACHE_DIR
    / "test_query_embeddings.json"
)

test_embedding_fingerprint = hashlib.sha256(
    (
        TEST_ID_HASH
        + frame_fingerprint(
            official_test_df,
            ["source_id", "source_text"],
        )
        + embedding_fingerprint
        + RETRIEVER_MODEL_NAME
    ).encode()
).hexdigest()

if (
    test_embedding_path.exists()
    and test_embedding_meta_path.exists()
    and json.loads(
        test_embedding_meta_path.read_text()
    )["fingerprint"] == test_embedding_fingerprint
):
    test_query_embeddings = np.load(
        test_embedding_path
    )
    print("Reused cached TEST embeddings.")
else:
    test_retriever = SentenceTransformer(
        RETRIEVER_MODEL_NAME,
        device="cpu",
    )

    test_query_embeddings = test_retriever.encode(
        official_test_df["source_text"].tolist(),
        batch_size=RETRIEVER_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    np.save(
        test_embedding_path,
        test_query_embeddings,
    )
    test_embedding_meta_path.write_text(
        json.dumps(
            {"fingerprint": test_embedding_fingerprint},
            indent=2,
        )
    )

    del test_retriever
    gc.collect()

country_indices = {
    key: group.index.to_numpy(dtype=np.int64)
    for key, group in retrieval_pool.groupby(
        "config",
        sort=False,
    )
}
country_domain_indices = {
    key: group.index.to_numpy(dtype=np.int64)
    for key, group in retrieval_pool.groupby(
        ["config", "domain"],
        sort=False,
    )
}

pool_dialect = (
    retrieval_pool["dialect"]
    .fillna("")
    .astype(str)
    .to_numpy()
)
pool_direction = (
    retrieval_pool["gender_direction"]
    .fillna("")
    .astype(str)
    .to_numpy()
)
pool_speaker = (
    retrieval_pool["speaker"]
    .fillna("")
    .astype(str)
    .to_numpy()
)
pool_conversation = (
    retrieval_pool["conversation_id"]
    .fillna("")
    .astype(str)
    .to_numpy()
)


def select_test_retrieved_shots(row, row_index, n=2):
    config = clean_string(row["config"])
    domain = clean_string(row["domain"])

    candidates = country_domain_indices.get(
        (config, domain)
    )

    if candidates is None or len(candidates) < n:
        candidates = country_indices.get(config)

    if candidates is None or len(candidates) < n:
        return test_random_map[str(row["source_id"])]

    different_conversation = (
        pool_conversation[candidates]
        != str(row["conversation_id"])
    )
    filtered = candidates[different_conversation]

    if len(filtered) >= n:
        candidates = filtered

    scores = (
        pool_embeddings[candidates]
        @ test_query_embeddings[row_index]
    )

    dialect = clean_string(row["dialect"])
    direction = clean_string(row["gender_direction"])
    speaker = clean_string(row["speaker"])

    if dialect:
        scores += (
            pool_dialect[candidates] == dialect
        ) * 0.05
    if direction:
        scores += (
            pool_direction[candidates] == direction
        ) * 0.03
    if speaker:
        scores += (
            pool_speaker[candidates] == speaker
        ) * 0.02

    ranked = candidates[np.argsort(-scores)]
    selected = []
    used_conversations = set()
    used_sources = set()

    for pool_index in ranked:
        candidate = retrieval_pool.iloc[
            int(pool_index)
        ]

        conversation_key = (
            str(candidate["config"]),
            str(candidate["conversation_id"]),
        )
        source_key = (
            str(candidate.get("pool_origin", "")),
            str(candidate["config"]),
            str(candidate["source_id"]),
        )

        if (
            conversation_key in used_conversations
            or source_key in used_sources
        ):
            continue

        selected.append(
            candidate[FEW_SHOT_COLUMNS].to_dict()
        )
        used_conversations.add(conversation_key)
        used_sources.add(source_key)

        if len(selected) == n:
            break

    if len(selected) < n:
        for pool_index in ranked:
            candidate = retrieval_pool.iloc[
                int(pool_index)
            ]
            source_key = (
                str(candidate.get("pool_origin", "")),
                str(candidate["config"]),
                str(candidate["source_id"]),
            )

            if source_key in used_sources:
                continue

            selected.append(
                candidate[FEW_SHOT_COLUMNS].to_dict()
            )
            used_sources.add(source_key)

            if len(selected) == n:
                break

    return selected


retrieved_cache_path = (
    TEST_CACHE_DIR
    / "selected_test_retrieved_shots.pkl"
)
retrieved_meta_path = (
    TEST_CACHE_DIR
    / "selected_test_retrieved_shots.json"
)

retrieved_fingerprint = hashlib.sha256(
    (
        test_embedding_fingerprint
        + embedding_fingerprint
        + "expanded_retrieval_policy_v1"
    ).encode()
).hexdigest()

if (
    retrieved_cache_path.exists()
    and retrieved_meta_path.exists()
    and json.loads(
        retrieved_meta_path.read_text()
    )["fingerprint"] == retrieved_fingerprint
):
    retrieved_shots_df = pd.read_pickle(
        retrieved_cache_path
    )
    print("Reused cached V03/V05 TEST shots.")
else:
    retrieved_rows = []

    for row_index, row in tqdm(
        official_test_df.iterrows(),
        total=len(official_test_df),
        desc="V03/V05 retrieved shots",
    ):
        retrieved_rows.append({
            "source_id": str(row["source_id"]),
            "few_shot_examples": (
                select_test_retrieved_shots(
                    row,
                    row_index,
                )
            ),
        })

    retrieved_shots_df = pd.DataFrame(
        retrieved_rows
    )
    retrieved_shots_df.to_pickle(
        retrieved_cache_path
    )
    retrieved_meta_path.write_text(
        json.dumps(
            {"fingerprint": retrieved_fingerprint},
            indent=2,
        )
    )

test_retrieved_map = dict(zip(
    retrieved_shots_df["source_id"].astype(str),
    retrieved_shots_df["few_shot_examples"],
))

SHOT_MAPS["test_random"] = test_random_map
SHOT_MAPS["test_retrieved"] = test_retrieved_map

if (
    set(test_random_map) != set(
        official_test_df["source_id"]
    )
    or set(test_retrieved_map) != set(
        official_test_df["source_id"]
    )
):
    raise RuntimeError(
        "TEST few-shot cache coverage mismatch."
    )

# All final variants use previous speaker labels.
def build_final_test_prompt(
    row,
    variant,
    shot_mode,
):
    examples = SHOT_MAPS[shot_mode][
        str(row["source_id"])
    ]

    example_text = "\n\n".join(
        format_example(example, i)
        for i, example in enumerate(
            examples,
            1,
        )
    )

    metadata = [
        f"Country/config: {clean_string(row['country'])}",
        f"Target dialect: {clean_string(row['dialect'])}",
        f"Domain: {clean_string(row['domain'])}",
    ]

    if (
        variant == "V05"
        and clean_string(row["participants"])
    ):
        metadata.append(
            "Persona/Roles: "
            f"{clean_string(row['participants'])}"
        )

    if clean_string(row["speaker"]):
        metadata.append(
            "Current speaker: "
            f"{clean_string(row['speaker'])}"
        )

    if clean_string(row["gender_direction"]):
        metadata.append(
            "Speaker-to-addressee gender direction: "
            f"{clean_string(row['gender_direction'])}"
        )

    return f"""### System:
{SYSTEM_PROMPT}

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{example_text}

Metadata:
{chr(10).join(metadata)}

Previous English dialogue context:
{format_previous_context(row, include_speakers=True)}

Current English turn:
{clean_string(row["source_text"])}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation.

### Arabic translation:
"""

# ------------------------------------------------------------
# Fingerprint and resume protection
# ------------------------------------------------------------

generation_manifest_path = (
    TEST_OUTPUT_DIR
    / "generation_manifest.json"
)

run_payload = {
    "system": TEST_SYSTEM_NAME,
    "test_id_hash": TEST_ID_HASH,
    "routes": FINAL_TEST_ROUTES,
    "random_shots": random_fingerprint,
    "retrieved_shots": retrieved_fingerprint,
    "generation": GENERATION_KWARGS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": GEN_BATCH_SIZE,
}

RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(
        run_payload,
        sort_keys=True,
        default=str,
    ).encode()
).hexdigest()


def atomic_json(payload, path):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    os.replace(temporary, path)


if TEST_PREDICTION_PATH.exists():
    if not generation_manifest_path.exists():
        raise RuntimeError(
            "Predictions exist without a generation manifest."
        )

    old_fingerprint = json.loads(
        generation_manifest_path.read_text()
    )["run_fingerprint"]

    if old_fingerprint != RUN_FINGERPRINT:
        raise RuntimeError(
            "Existing predictions belong to a different "
            "route/prompt configuration."
        )

atomic_json(
    {
        **run_payload,
        "run_fingerprint": RUN_FINGERPRINT,
    },
    generation_manifest_path,
)

if TEST_PREDICTION_PATH.exists():
    predictions_df = pd.read_csv(
        TEST_PREDICTION_PATH,
        dtype={"source_id": str},
    )
else:
    predictions_df = pd.DataFrame(
        columns=[
            "source_id",
            "prediction",
            "country",
            "adapter_key",
            "variant",
            "shot_mode",
        ]
    )

valid_existing = (
    predictions_df["prediction"].notna()
    & predictions_df["prediction"]
      .astype(str)
      .str.strip()
      .ne("")
)

predictions_df = predictions_df[
    valid_existing
].drop_duplicates(
    "source_id",
    keep="last",
)

completed_ids = set(
    predictions_df["source_id"].astype(str)
)

print(
    f"Resuming with {len(completed_ids):,}/"
    f"{EXPECTED_TEST_TURNS:,} valid predictions."
)

# ------------------------------------------------------------
# Grouped inference: adapter switch only between groups
# ------------------------------------------------------------

current_adapter = None

for group in FINAL_TEST_GROUPS:
    adapter = group["adapter"]
    variant = group["variant"]
    shot_mode = group["shot_mode"]
    countries = group["countries"]

    group_rows = (
        official_test_df[
            official_test_df["config"].isin(countries)
            & ~official_test_df["source_id"].isin(
                completed_ids
            )
        ]
        .sort_values("_test_row_idx")
        .copy()
    )

    if group_rows.empty:
        print(
            f"Already complete: {adapter} + "
            f"{variant} → {countries}"
        )
        continue

    if adapter != current_adapter:
        activate_adapter(adapter)
        current_adapter = adapter

    print(
        f"\n{adapter} + {variant} → {countries}: "
        f"{len(group_rows):,} remaining"
    )

    pending = []

    for start in tqdm(
        range(0, len(group_rows), GEN_BATCH_SIZE),
        desc=f"{adapter}-{variant}",
    ):
        batch = group_rows.iloc[
            start:start + GEN_BATCH_SIZE
        ]

        prompts = [
            build_final_test_prompt(
                row,
                variant,
                shot_mode,
            )
            for _, row in batch.iterrows()
        ]

        generated = generate_batch(prompts)

        for (_, row), prediction in zip(
            batch.iterrows(),
            generated,
        ):
            pending.append({
                "source_id": str(row["source_id"]),
                "prediction": prediction,
                "country": str(row["config"]),
                "adapter_key": adapter,
                "variant": variant,
                "shot_mode": shot_mode,
            })

        if len(pending) >= SAVE_EVERY:
            predictions_df = pd.concat(
                [
                    predictions_df,
                    pd.DataFrame(pending),
                ],
                ignore_index=True,
            ).drop_duplicates(
                "source_id",
                keep="last",
            )

            atomic_csv(
                predictions_df,
                TEST_PREDICTION_PATH,
            )

            completed_ids.update(
                x["source_id"] for x in pending
            )
            pending = []

    if pending:
        predictions_df = pd.concat(
            [
                predictions_df,
                pd.DataFrame(pending),
            ],
            ignore_index=True,
        ).drop_duplicates(
            "source_id",
            keep="last",
        )

        atomic_csv(
            predictions_df,
            TEST_PREDICTION_PATH,
        )

        completed_ids.update(
            x["source_id"] for x in pending
        )

# ------------------------------------------------------------
# Exact prediction validation
# ------------------------------------------------------------

expected_ids = set(
    official_test_df["source_id"].astype(str)
)
actual_ids = set(
    predictions_df["source_id"].astype(str)
)

if actual_ids != expected_ids:
    raise RuntimeError(
        f"Prediction coverage failure: "
        f"missing={len(expected_ids - actual_ids)}, "
        f"extra={len(actual_ids - expected_ids)}"
    )

if predictions_df["source_id"].duplicated().any():
    raise RuntimeError("Duplicate prediction IDs.")

if (
    predictions_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError("Empty predictions remain.")

submission_turn_df = (
    official_test_df[
        [
            "source_id",
            "config",
            "conversation_id",
            "turn_order",
            "_test_row_idx",
        ]
    ]
    .merge(
        predictions_df[
            [
                "source_id",
                "prediction",
                "adapter_key",
                "variant",
                "shot_mode",
            ]
        ],
        on="source_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("_test_row_idx")
    .reset_index(drop=True)
)

atomic_csv(
    submission_turn_df,
    TEST_PREDICTION_PATH,
)

# ------------------------------------------------------------
# Generation diagnostics — not official TEST metrics
# ------------------------------------------------------------

def arabic_letter_ratio(text):
    letters = [
        c for c in str(text)
        if c.isalpha()
    ]

    if not letters:
        return 0.0

    return sum(
        "\u0600" <= c <= "\u06ff"
        for c in letters
    ) / len(letters)


submission_turn_df["prediction_chars"] = (
    submission_turn_df["prediction"]
    .astype(str)
    .str.len()
)
submission_turn_df["arabic_letter_ratio"] = (
    submission_turn_df["prediction"]
    .map(arabic_letter_ratio)
)

test_diagnostics_df = (
    submission_turn_df
    .groupby("config", as_index=False)
    .agg(
        conversations=("conversation_id", "nunique"),
        turns=("source_id", "size"),
        mean_prediction_chars=(
            "prediction_chars",
            "mean",
        ),
        mean_arabic_letter_ratio=(
            "arabic_letter_ratio",
            "mean",
        ),
    )
)

atomic_csv(
    test_diagnostics_df,
    TEST_OUTPUT_DIR
    / "generation_diagnostics.csv",
)

# ------------------------------------------------------------
# Official predictions.jsonl
# ------------------------------------------------------------

ordered_submission_df = (
    submission_turn_df
    .sort_values(
        ["config", "conversation_id", "turn_order"]
    )
    .reset_index(drop=True)
)

submission_records = []

for (
    config,
    conversation_id,
), conversation_df in ordered_submission_df.groupby(
    ["config", "conversation_id"],
    sort=True,
):
    conversation_df = conversation_df.sort_values(
        "turn_order"
    )

    if conversation_df["turn_order"].duplicated().any():
        raise RuntimeError(
            f"Duplicate turn order: "
            f"{config}/{conversation_id}"
        )

    submission_records.append({
        "conv_id": str(conversation_id),
        "country": str(config),
        "turns": [
            {
                "turn_order": int(row.turn_order),
                "prediction": str(row.prediction),
            }
            for row in conversation_df.itertuples()
        ],
    })

submission_turn_count = sum(
    len(record["turns"])
    for record in submission_records
)

if submission_turn_count != EXPECTED_TEST_TURNS:
    raise RuntimeError(
        f"Submission has {submission_turn_count:,} turns."
    )

temporary_jsonl = Path(
    str(TEST_JSONL_PATH) + ".tmp"
)

with open(
    temporary_jsonl,
    "w",
    encoding="utf-8",
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

os.replace(
    temporary_jsonl,
    TEST_JSONL_PATH,
)

temporary_zip = Path(
    str(TEST_ZIP_PATH) + ".tmp"
)

with zipfile.ZipFile(
    temporary_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        TEST_JSONL_PATH,
        arcname="predictions.jsonl",
    )

os.replace(
    temporary_zip,
    TEST_ZIP_PATH,
)

# ------------------------------------------------------------
# Exact ZIP readback validation
# ------------------------------------------------------------

readback_keys = set()
readback_turns = 0
readback_conversations = 0

with zipfile.ZipFile(
    TEST_ZIP_PATH,
    "r",
) as archive:
    if archive.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only predictions.jsonl."
        )

    with archive.open(
        "predictions.jsonl",
        "r",
    ) as file:
        for binary_line in file:
            record = json.loads(
                binary_line.decode("utf-8")
            )
            readback_conversations += 1

            for turn in record["turns"]:
                key = (
                    str(record["country"]),
                    str(record["conv_id"]),
                    int(turn["turn_order"]),
                )

                if key in readback_keys:
                    raise RuntimeError(
                        f"Duplicate ZIP key: {key}"
                    )

                readback_keys.add(key)
                readback_turns += 1

expected_keys = set(zip(
    official_test_df["config"].astype(str),
    official_test_df[
        "conversation_id"
    ].astype(str),
    official_test_df["turn_order"].astype(int),
))

if readback_keys != expected_keys:
    raise RuntimeError(
        "ZIP keys do not match official TEST."
    )

if readback_turns != EXPECTED_TEST_TURNS:
    raise RuntimeError("ZIP turn-count mismatch.")

if (
    readback_conversations
    != EXPECTED_TEST_CONVERSATIONS
):
    raise RuntimeError(
        "ZIP conversation-count mismatch."
    )

submission_manifest = {
    "system": TEST_SYSTEM_NAME,
    "run_fingerprint": RUN_FINGERPRINT,
    "dataset_revision": (
        "codabench_private_test_807229"
    ),
    "test_id_hash": TEST_ID_HASH,
    "test_turns": readback_turns,
    "conversations": readback_conversations,
    "routes": FINAL_TEST_ROUTES,
    "beam": 4,
    "selection_macro_spBLEU": (
        SELECTION_MACRO_SPBLEU
    ),
    "selection_macro_chrF++": (
        SELECTION_MACRO_CHRFPP
    ),
    "official_test_metrics": (
        "Unavailable locally: private TEST "
        "references are not released."
    ),
    "jsonl_path": str(TEST_JSONL_PATH),
    "zip_path": str(TEST_ZIP_PATH),
    "completed_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

atomic_json(
    submission_manifest,
    TEST_OUTPUT_DIR
    / "submission_manifest.json",
)

print("\n" + "=" * 90)
print("OFFICIAL TEST SUBMISSION READY")
print("=" * 90)
print("Turns:", readback_turns)
print("Conversations:", readback_conversations)
print("Countries:", sorted(COUNTRIES))
print(
    "Locked-40 validation estimate:",
    f"{SELECTION_MACRO_SPBLEU:.6f} spBLEU | "
    f"{SELECTION_MACRO_CHRFPP:.6f} chrF++",
)
print(
    "Official TEST metrics: available only "
    "after Codabench submission."
)
print("\nSUBMIT THIS ZIP:")
print(TEST_ZIP_PATH)

display(test_diagnostics_df)
display(FileLink(str(TEST_ZIP_PATH)))

V01 deterministic shots:   0%|          | 0/14459 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

V03/V05 retrieved shots:   0%|          | 0/14459 [00:00<?, ?it/s]

Resuming with 0/14,459 valid predictions.

cont_5200 + V01 → ['JO', 'LB', 'PS', 'SD']: 4,245 remaining


cont_5200-V01:   0%|          | 0/2123 [00:00<?, ?it/s]

...


cont_7600 + V01 → ['TN']: 1,114 remaining


cont_7600-V01:   0%|          | 0/557 [00:00<?, ?it/s]


OFFICIAL TEST SUBMISSION READY
Turns: 14459
Conversations: 4673
Countries: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
Locked-40 validation estimate: 30.182420 spBLEU | 44.805134 chrF++
Official TEST metrics: available only after Codabench submission.

SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/100_official_private_test_continuation_router_threshold015_beam4_v1/submission_predictions.zip


,config,conversations,turns,mean_prediction_chars,mean_arabic_letter_ratio
0,EG,359,1113,75.078167,0.994014
1,JO,348,1109,69.574391,0.996866
2,LB,368,1110,71.672072,0.978912
3,LY,423,1309,76.292590,0.997621
4,MA,362,1111,84.100810,0.965061
5,MR,365,1119,74.405719,0.997001
6,OM,370,1107,74.224029,0.998786
7,PS,351,1111,73.464446,0.998368
8,SA,359,1114,78.052065,0.998566
9,SD,283,915,74.312568,0.997675


/home/mabdallah/alexandriax_mt_14d/inference_variants/100_official_private_test_continuation_router_threshold015_beam4_v1/submission_predictions.zip

### **New Experiment**